# Notebook 03: Multi-Agent Predictive Maintenance System

Bu notebook, projenin en kritik bileşeni olan **Çoklu Ajan (Multi-Agent)** mimarisini kurar.

### Neden Multi-Agent?
Notebook 01'de tek bir LLM çağrısıyla analiz yaptık. Bu yaklaşım çalışır, ancak gerçek dünyada yetersizdir çünkü:
- Tek bir prompt'a her şeyi yüklemek (veri analizi + bilgi getirme + karar verme) LLM'in odağını dağıtır.
- Hata ayıklama zorlaşır — hangi adımda sorun çıktığını bilemezsiniz.
- Ölçeklenebilirlik sıfırdır — yeni bir yetenek eklemek tüm prompt'u yeniden yazmak demektir.

### Mimarimiz: 3 Uzman Ajan
| Ajan | Rol | Girdi | Çıktı |
| :--- | :--- | :--- | :--- |
| **SensorAnalyst** | Ham sensör verisini analiz eder, anomalileri tespit eder | Sensör verisi (dict) | Anomali raporu (dict) |
| **KnowledgeRetriever** | ChromaDB'den ilgili teknik dokümanları getirir | Anomali raporu | İlgili kılavuz bölümleri (str) |
| **MaintenancePlanner** | İlk iki ajanın çıktılarını birleştirip final rapor üretir | Anomali raporu + Kılavuz | Profesyonel bakım raporu (str) |

```
Sensör Verisi → [SensorAnalyst] → Anomali Raporu → [KnowledgeRetriever] → İlgili Dokümanlar
                                        ↓                                        ↓
                                  [MaintenancePlanner] ← ← ← ← ← ← ← ← ← ← ← ←
                                        ↓
                                  Final Bakım Raporu
```

## 3.1. Kütüphaneler ve Konfigürasyon
Bu bölümde tüm bağımlılıkları yükleyip API bağlantısını kuruyoruz.
Groq API ücretsiz tier kullanılmaktadır (günde 1000 istek, dakikada 30 istek).

In [3]:
# === Temel Kütüphaneler ===
import pandas as pd
import numpy as np
import requests
import json
import time
import os
from datetime import datetime

# === Vector Database ===
import chromadb
from chromadb.utils import embedding_functions

print("✅ Kütüphaneler başarıyla yüklendi.")

✅ Kütüphaneler başarıyla yüklendi.


In [5]:
# =============================================
# API & VERİTABANI KONFİGÜRASYONU
# =============================================

# Groq API Ayarları
GROQ_API_KEY = "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"  # <-- Kendi Groq API anahtarınızı yazın
GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

# ChromaDB Bağlantısı (Notebook 02'de oluşturulan veritabanını kullanıyoruz)
local_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
chroma_client = chromadb.PersistentClient(path="../data/db")
collection = chroma_client.get_or_create_collection(
    name="technical_manual",
    embedding_function=local_ef
)

print(f"✅ Groq API yapılandırıldı. Model: {GROQ_MODEL}")
print(f"✅ ChromaDB bağlantısı kuruldu. Kayıt sayısı: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Groq API yapılandırıldı. Model: llama-3.3-70b-versatile
✅ ChromaDB bağlantısı kuruldu. Kayıt sayısı: 10


In [7]:
# =============================================
# VERİ YÜKLEME
# =============================================

df = pd.read_csv('../data/raw/predictive_maintenance_dataset.csv')
print(f"✅ Veri seti yüklendi: {df.shape[0]} satır, {df.shape[1]} sütun")
print(f"   Arızalı kayıt sayısı: {df[df['label']==1].shape[0]}")
print(f"   Normal kayıt sayısı: {df[df['label']==0].shape[0]}")

✅ Veri seti yüklendi: 1800 satır, 10 sütun
   Arızalı kayıt sayısı: 202
   Normal kayıt sayısı: 1598


## 3.2. LLM İletişim Katmanı (Groq Gateway)

Tüm ajanların LLM ile iletişimi tek bir fonksiyon üzerinden geçer.
Bu **Gateway Pattern** sayesinde:
- Rate limiting merkezi olarak yönetilir
- Hata yönetimi tek noktadan yapılır
- İleride farklı bir LLM provider'a geçmek tek satır değişiklik gerektirir

In [10]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """
    Merkezi LLM Gateway fonksiyonu.
    Tüm ajanlar bu fonksiyon üzerinden Groq API'ye erişir.
    
    Args:
        system_prompt: Ajanın rolünü ve davranışını tanımlayan sistem mesajı
        user_prompt: Ajana gönderilen asıl görev/soru
        temperature: Yanıt çeşitliliği (0.0=deterministik, 1.0=yaratıcı)
    
    Returns:
        LLM'in ürettiği metin yanıtı
    """
    time.sleep(2)  # Rate limit koruması (Groq free: 30 req/dk)
    
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": GROQ_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": 2000
    }
    
    try:
        response = requests.post(GROQ_URL, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            error_msg = response.json().get('error', {}).get('message', 'Bilinmeyen hata')
            return f"❌ API Hatası ({response.status_code}): {error_msg}"
    except Exception as e:
        return f"❌ Bağlantı Hatası: {str(e)}"

# --- Gateway Testi ---
test_result = call_llm(
    system_prompt="Sen bir test asistanısın.",
    user_prompt="Bu bir bağlantı testidir. 'BAĞLANTI BAŞARILI' yaz."
)
print(test_result)

BAĞLANTI BAŞARILI


## 3.3. Ajan 1: SensorAnalyst (Sensör Analiz Uzmanı)

Bu ajan **LLM kullanmaz** — tamamen kural tabanlı (rule-based) çalışır.
Neden? Çünkü eşik değerleri kesindir ve bir LLM'in "yorumlamasına" bırakılmamalıdır.

Bu, gerçek dünyada da standart bir yaklaşımdır:
- Deterministik kararlar → Kod ile
- Yorumlama ve doğal dil üretimi → LLM ile

In [13]:
class SensorAnalystAgent:
    """
    Ajan 1: Sensör Verisi Analiz Uzmanı
    
    Görevi: Ham sensör verisini alır, teknik kılavuzdaki eşik değerlerine göre
    her parametrenin durumunu (Normal/Uyarı/Kritik) belirler ve olası arıza
    tiplerini tespit eder.
    
    Bu ajan deterministik (kural tabanlı) çalışır — LLM kullanmaz.
    """
    
    # Teknik kılavuzdan alınan eşik değerleri (Bölüm 2: Operasyonel KPI'lar)
    THRESHOLDS = {
        'vibration':   {'normal': (0.60, 0.90), 'warning': (1.10, 1.25), 'critical': 1.30},
        'acoustic':    {'normal': (0.50, 0.70), 'warning': (0.85, 0.95), 'critical': 1.00},
        'temperature': {'normal': (60, 72),     'warning': (76, 82),     'critical': 85},
        'current':     {'normal': (11.5, 13.5), 'warning': (14.5, 15.2), 'critical': 15.5}
    }
    
    # Arıza modları (Bölüm 3: RCA)
    FAILURE_MODES = {
        'Tip_A': {
            'name': 'Mekanik Yorgunluk ve Rulman Arızası',
            'conditions': lambda d: d['vibration'] > 1.25 and d['acoustic'] > 0.90
        },
        'Tip_B': {
            'name': 'Elektriksel Aşırı Yük ve Isınma (Overload)',
            'conditions': lambda d: d['current'] > 15.0 and d['temperature'] > 80
        },
        'Tip_C': {
            'name': 'Akustik Rezonans (Gevşek Parça)',
            'conditions': lambda d: d['acoustic'] > 1.10 and d['vibration'] < 1.10 and d['temperature'] < 76
        }
    }
    
    def _classify_parameter(self, param_name: str, value: float) -> str:
        """Tek bir parametrenin durumunu sınıflandırır."""
        t = self.THRESHOLDS[param_name]
        if value >= t['critical']:
            return 'CRITICAL'
        elif t['warning'][0] <= value <= t['warning'][1]:
            return 'WARNING'
        elif value > t['normal'][1]:  # Normal üstü ama warning altı
            return 'ELEVATED'
        else:
            return 'NORMAL'
    
    def analyze(self, sensor_data: dict) -> dict:
        """
        Ana analiz fonksiyonu.
        
        Args:
            sensor_data: Sensör verisi dictionary (timestamp, machine_id, vibration, vb.)
        
        Returns:
            Yapılandırılmış anomali raporu (dict)
        """
        # 1. Her parametreyi sınıflandır
        param_status = {}
        for param in ['vibration', 'acoustic', 'temperature', 'current']:
            value = sensor_data[param]
            status = self._classify_parameter(param, value)
            threshold = self.THRESHOLDS[param]
            param_status[param] = {
                'value': value,
                'status': status,
                'normal_range': f"{threshold['normal'][0]} - {threshold['normal'][1]}",
                'critical_threshold': threshold['critical']
            }
        
        # 2. Arıza modlarını kontrol et
        detected_failures = []
        for mode_id, mode_info in self.FAILURE_MODES.items():
            if mode_info['conditions'](sensor_data):
                detected_failures.append({
                    'mode': mode_id,
                    'name': mode_info['name']
                })
        
        # 3. IMF analizi
        imf_alert = None
        if sensor_data.get('IMF_1', 0) > 0.30:
            imf_alert = f"IMF_1 değeri ({sensor_data['IMF_1']}) > 0.30 — Yapısal çatlak riski!"
        
        # 4. Genel risk seviyesi
        statuses = [p['status'] for p in param_status.values()]
        if 'CRITICAL' in statuses:
            overall_risk = 'CRITICAL'
        elif 'WARNING' in statuses:
            overall_risk = 'WARNING'
        elif 'ELEVATED' in statuses:
            overall_risk = 'ELEVATED'
        else:
            overall_risk = 'NORMAL'
        
        # 5. Rapor oluştur
        report = {
            'timestamp': sensor_data.get('timestamp', 'N/A'),
            'machine_id': sensor_data.get('machine_id', 'N/A'),
            'overall_risk': overall_risk,
            'parameter_analysis': param_status,
            'detected_failure_modes': detected_failures,
            'imf_alert': imf_alert
        }
        
        return report

print("✅ SensorAnalystAgent tanımlandı.")

✅ SensorAnalystAgent tanımlandı.


### SensorAnalyst Test

In [16]:
# Test: Arızalı bir kayıtla deneyelim
sensor_agent = SensorAnalystAgent()

# Veri setinden arızalı ilk kaydı al
failure_sample = df[df['label'] == 1].iloc[0].to_dict()
print("Girdi (Ham Sensör Verisi):")
print(json.dumps(failure_sample, indent=2, default=str))

# Analiz et
anomaly_report = sensor_agent.analyze(failure_sample)

print("\n" + "="*60)
print("Çıktı (SensorAnalyst Raporu):")
print("="*60)
print(f"Makine: {anomaly_report['machine_id']}")
print(f"Zaman: {anomaly_report['timestamp']}")
print(f"Genel Risk: 🔴 {anomaly_report['overall_risk']}")
print(f"\nParametre Durumları:")
for param, info in anomaly_report['parameter_analysis'].items():
    emoji = {'CRITICAL': '🔴', 'WARNING': '🟡', 'ELEVATED': '🟠', 'NORMAL': '🟢'}[info['status']]
    print(f"  {emoji} {param}: {info['value']} ({info['status']}) | Normal: {info['normal_range']}")
print(f"\nTespit Edilen Arıza Modları:")
for f in anomaly_report['detected_failure_modes']:
    print(f"  ⚠️  {f['mode']}: {f['name']}")
if anomaly_report['imf_alert']:
    print(f"\n🔬 IMF Uyarısı: {anomaly_report['imf_alert']}")

Girdi (Ham Sensör Verisi):
{
  "timestamp": "2024-07-01 08:01:00",
  "machine_id": "M01",
  "vibration": 1.398,
  "acoustic": 0.834,
  "temperature": 76.2,
  "current": 15.08,
  "IMF_1": 0.345,
  "IMF_2": 0.132,
  "IMF_3": 0.001,
  "label": 1
}

Çıktı (SensorAnalyst Raporu):
Makine: M01
Zaman: 2024-07-01 08:01:00
Genel Risk: 🔴 CRITICAL

Parametre Durumları:
  🔴 vibration: 1.398 (CRITICAL) | Normal: 0.6 - 0.9
  🟠 acoustic: 0.834 (ELEVATED) | Normal: 0.5 - 0.7
  🟡 temperature: 76.2 (WARNING) | Normal: 60 - 72
  🟡 current: 15.08 (WARNING) | Normal: 11.5 - 13.5

Tespit Edilen Arıza Modları:

🔬 IMF Uyarısı: IMF_1 değeri (0.345) > 0.30 — Yapısal çatlak riski!


## 3.4. Ajan 2: KnowledgeRetriever (Bilgi Getirme Uzmanı)

Bu ajan, SensorAnalyst'in tespit ettiği anomalilere göre ChromaDB'den 
**en ilgili teknik doküman parçalarını** getirir.

Notebook 02'de kurduğumuz vektör veritabanını burada kullanıyoruz.
Bu ajan da LLM kullanmaz — semantic search (anlamsal arama) ile çalışır.

In [19]:
class KnowledgeRetrieverAgent:
    """
    Ajan 2: Teknik Bilgi Getirme Uzmanı (RAG Retrieval)
    
    Görevi: SensorAnalyst'in anomali raporuna göre ChromaDB'den
    ilgili teknik kılavuz bölümlerini getirir.
    
    Bu ajan semantic search (vektör benzerliği) kullanır — LLM kullanmaz.
    """
    
    def __init__(self, chroma_collection):
        self.collection = chroma_collection
    
    def _build_query(self, anomaly_report: dict) -> str:
        """
        Anomali raporundan akıllı bir arama sorgusu oluşturur.
        Bu sorgu, vektör DB'de en ilgili bölümleri bulmak için kullanılır.
        """
        query_parts = []
        
        # Tespit edilen arıza modlarını sorguya ekle
        for failure in anomaly_report.get('detected_failure_modes', []):
            query_parts.append(failure['name'])
        
        # Kritik/Uyarı durumundaki parametreleri ekle
        for param, info in anomaly_report.get('parameter_analysis', {}).items():
            if info['status'] in ['CRITICAL', 'WARNING']:
                query_parts.append(f"{param} {info['status'].lower()} threshold exceeded")
        
        # IMF uyarısı varsa ekle
        if anomaly_report.get('imf_alert'):
            query_parts.append("IMF structural crack analysis")
        
        # Sorguyu birleştir
        query = ". ".join(query_parts) if query_parts else "machine maintenance general procedures"
        return query
    
    def retrieve(self, anomaly_report: dict, n_results: int = 3) -> dict:
        """
        Ana bilgi getirme fonksiyonu.
        
        Args:
            anomaly_report: SensorAnalyst'in ürettiği anomali raporu
            n_results: Getirilecek doküman parçası sayısı
        
        Returns:
            Getirilen dokümanlar ve metadata (dict)
        """
        # 1. Akıllı sorgu oluştur
        query = self._build_query(anomaly_report)
        
        # 2. ChromaDB'de ara
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        
        # 3. Sonuçları yapılandır
        retrieved_docs = []
        if results['documents'][0]:
            for i, doc in enumerate(results['documents'][0]):
                retrieved_docs.append({
                    'content': doc,
                    'relevance_rank': i + 1,
                    'metadata': results['metadatas'][0][i] if results['metadatas'][0] else {}
                })
        
        return {
            'query_used': query,
            'num_results': len(retrieved_docs),
            'documents': retrieved_docs,
            'combined_context': "\n\n---\n\n".join([d['content'] for d in retrieved_docs])
        }

print("✅ KnowledgeRetrieverAgent tanımlandı.")

✅ KnowledgeRetrieverAgent tanımlandı.


### KnowledgeRetriever Test

In [22]:
# Test: SensorAnalyst'in çıktısını KnowledgeRetriever'a verelim
knowledge_agent = KnowledgeRetrieverAgent(collection)

retrieval_result = knowledge_agent.retrieve(anomaly_report)

print("Oluşturulan Arama Sorgusu:")
print(f"  → {retrieval_result['query_used']}")
print(f"\nBulunan İlgili Doküman Sayısı: {retrieval_result['num_results']}")
print("\n" + "="*60)
for doc in retrieval_result['documents']:
    print(f"\n[Doküman #{doc['relevance_rank']}]:")
    print(doc['content'][:300] + "..." if len(doc['content']) > 300 else doc['content'])
    print("-"*40)

Oluşturulan Arama Sorgusu:
  → vibration critical threshold exceeded. temperature warning threshold exceeded. current warning threshold exceeded. IMF structural crack analysis

Bulunan İlgili Doküman Sayısı: 3


[Doküman #1]:
2. OPERASYONEL EŞİK DEĞERLERİ (KPI)



| Parametre | Normal Aralık | Uyarı (Warning) | Kritik (Critical) |

| :--- | :--- | :--- | :--- |

| Vibration | 0.60 - 0.90 | 1.10 - 1.25 | > 1.30 |

| Temperature | 60°C - 72°C | 76°C - 82°C | > 85°C |

| Current | 11.5A - 13.5A | 14.5A - 15.2A | > 15.5A |

...
----------------------------------------

[Doküman #2]:
# 1.1. Sensör Spesifikasyonları

\* \*\*Vibration (mm/s^2):\*\* Gövde üzerindeki ivmeölçerden gelir. Mekanik dengesizliği ölçer.

\* \*\*Acoustic (dB):\*\* Rulman yataklarındaki yüksek frekanslı sürtünme seslerini takip eder.

\* \*\*Temperature (°C):\*\* Motor sargı ve yüzey sıcaklığıdır.

\* \*\*C...
----------------------------------------

[Doküman #3]:
4. GELİŞMİŞ ANALİZ NOTLARI (IMF ANALİZİ)

\* \*\*IMF\_

## 3.5. Ajan 3: MaintenancePlanner (Bakım Planlama Uzmanı)

Bu ajan **LLM kullanan tek ajandır**. İlk iki ajanın yapılandırılmış çıktılarını alır
ve bunları profesyonel bir bakım raporu haline getirir.

Bu tasarım kasıtlıdır (by design):
- Ajan 1 ve 2 deterministik ve hızlıdır (milisaniyeler)
- Sadece Ajan 3 LLM çağrısı yapar (maliyet ve zaman tasarrufu)
- LLM'e "ham veri" yerine "ön-işlenmiş, yapılandırılmış bilgi" gönderilir → daha kaliteli çıktı

In [25]:
class MaintenancePlannerAgent:
    """
    Ajan 3: Bakım Planlama Uzmanı (LLM-Powered)
    
    Görevi: SensorAnalyst ve KnowledgeRetriever'ın çıktılarını sentezleyerek
    profesyonel bir bakım ve müdahale raporu üretir.
    
    Bu ajan Groq API üzerinden Llama 3.3 70B modelini kullanır.
    """
    
    SYSTEM_PROMPT = """Sen Alpha-V8 endüstriyel makineler konusunda 20 yıllık deneyime sahip 
kıdemli bir Bakım Mühendisisin. Görevin, sensör analizlerini ve teknik kılavuz bilgilerini
birleştirerek net, uygulanabilir bakım raporları üretmektir.

Raporlarında şu yapıyı takip et:
1. DURUM ÖZETİ: Makinenin genel durumu (1-2 cümle)
2. TESPİT EDİLEN SORUNLAR: Her anomali için detaylı açıklama
3. KÖK NEDEN ANALİZİ: Olası kök nedenler
4. ACİL MÜDAHALE PLANI: Öncelikli adımlar (numaralı liste)
5. ÖNLEYİCİ BAKIM ÖNERİLERİ: Uzun vadeli öneriler

Teknik ve profesyonel bir dil kullan. Belirsiz durumlarda "muhtemel" veya "olası" gibi
niteleyiciler ekle."""
    
    def _format_anomaly_summary(self, anomaly_report: dict) -> str:
        """Anomali raporunu LLM'in anlayacağı metin formatına çevirir."""
        lines = []
        lines.append(f"Makine: {anomaly_report['machine_id']}")
        lines.append(f"Zaman: {anomaly_report['timestamp']}")
        lines.append(f"Genel Risk Seviyesi: {anomaly_report['overall_risk']}")
        lines.append("\nParametre Durumları:")
        
        for param, info in anomaly_report['parameter_analysis'].items():
            lines.append(f"  - {param}: {info['value']} → Durum: {info['status']} "
                        f"(Normal aralık: {info['normal_range']}, Kritik eşik: {info['critical_threshold']})")
        
        if anomaly_report['detected_failure_modes']:
            lines.append("\nTespit Edilen Arıza Modları:")
            for f in anomaly_report['detected_failure_modes']:
                lines.append(f"  - {f['mode']}: {f['name']}")
        
        if anomaly_report.get('imf_alert'):
            lines.append(f"\nIMF Uyarısı: {anomaly_report['imf_alert']}")
        
        return "\n".join(lines)
    
    def generate_report(self, anomaly_report: dict, retrieved_context: str) -> str:
        """
        Final bakım raporunu üretir.
        
        Args:
            anomaly_report: SensorAnalyst'in ürettiği anomali raporu
            retrieved_context: KnowledgeRetriever'ın getirdiği teknik dokümanlar
        
        Returns:
            Profesyonel bakım raporu (str)
        """
        anomaly_summary = self._format_anomaly_summary(anomaly_report)
        
        user_prompt = f"""Aşağıdaki sensör analiz sonuçlarını ve teknik kılavuz bilgilerini 
kullanarak kapsamlı bir bakım raporu hazırla.

[SENSÖR ANALİZ SONUÇLARI (Ajan 1 Çıktısı)]:
{anomaly_summary}

[TEKNİK KILAVUZDAN İLGİLİ BÖLÜMLER (Ajan 2 Çıktısı)]:
{retrieved_context}

Yukarıdaki yapıya uygun bir bakım raporu üret."""
        
        return call_llm(self.SYSTEM_PROMPT, user_prompt, temperature=0.2)

print("✅ MaintenancePlannerAgent tanımlandı.")

✅ MaintenancePlannerAgent tanımlandı.


## 3.6. Orkestratör: Multi-Agent Pipeline

Bu sınıf, 3 ajanı sıralı bir pipeline'da çalıştırır ve her adımın
çıktısını bir sonraki ajana aktarır. Ayrıca her adımın süresini ölçer
ve tam bir çalışma günlüğü (execution log) tutar.

```
Pipeline Akışı:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Sensör Verisi
       ↓
  [1. SensorAnalyst]     → Anomali Raporu (dict)
       ↓
  [2. KnowledgeRetriever] → İlgili Dokümanlar (str)
       ↓
  [3. MaintenancePlanner] → Final Bakım Raporu (str)
       ↓
  Yapılandırılmış Çıktı (tüm ara adımlar + final rapor)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

In [28]:
class MultiAgentOrchestrator:
    """
    3 ajanı koordine eden merkezi orkestratör.
    Her adımın süresini ölçer ve execution log tutar.
    """
    
    def __init__(self, chroma_collection):
        self.sensor_agent = SensorAnalystAgent()
        self.knowledge_agent = KnowledgeRetrieverAgent(chroma_collection)
        self.planner_agent = MaintenancePlannerAgent()
        self.execution_log = []
    
    def _log_step(self, agent_name: str, duration: float, status: str):
        """Her adımı loga kaydeder."""
        self.execution_log.append({
            'agent': agent_name,
            'duration_sec': round(duration, 3),
            'status': status,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
    
    def run(self, sensor_data: dict) -> dict:
        """
        Tam pipeline'ı çalıştırır.
        
        Args:
            sensor_data: Ham sensör verisi (dict)
        
        Returns:
            Tüm ara adımları ve final raporu içeren sonuç (dict)
        """
        self.execution_log = []  # Logu sıfırla
        pipeline_start = time.time()
        
        # ===== ADIM 1: Sensör Analizi =====
        print("🔵 Adım 1/3: SensorAnalyst çalışıyor...")
        t0 = time.time()
        anomaly_report = self.sensor_agent.analyze(sensor_data)
        self._log_step('SensorAnalyst', time.time() - t0, 'OK')
        print(f"   ✅ Tamamlandı ({time.time()-t0:.3f}s) → Risk: {anomaly_report['overall_risk']}")
        
        # ===== ADIM 2: Bilgi Getirme =====
        print("🟡 Adım 2/3: KnowledgeRetriever çalışıyor...")
        t0 = time.time()
        retrieval_result = self.knowledge_agent.retrieve(anomaly_report)
        self._log_step('KnowledgeRetriever', time.time() - t0, 'OK')
        print(f"   ✅ Tamamlandı ({time.time()-t0:.3f}s) → {retrieval_result['num_results']} doküman bulundu")
        
        # ===== ADIM 3: Rapor Üretimi (LLM) =====
        print("🔴 Adım 3/3: MaintenancePlanner çalışıyor (LLM çağrısı)...")
        t0 = time.time()
        final_report = self.planner_agent.generate_report(
            anomaly_report, 
            retrieval_result['combined_context']
        )
        self._log_step('MaintenancePlanner', time.time() - t0, 'OK')
        print(f"   ✅ Tamamlandı ({time.time()-t0:.3f}s)")
        
        total_time = time.time() - pipeline_start
        print(f"\n🏁 Pipeline tamamlandı! Toplam süre: {total_time:.2f}s")
        
        return {
            'input': sensor_data,
            'anomaly_report': anomaly_report,
            'retrieval_result': retrieval_result,
            'final_report': final_report,
            'execution_log': self.execution_log,
            'total_duration_sec': round(total_time, 2)
        }

print("✅ MultiAgentOrchestrator tanımlandı.")

✅ MultiAgentOrchestrator tanımlandı.


## 3.7. Pipeline Testi: Tek Arızalı Kayıt

In [31]:
# Orkestratörü başlat
orchestrator = MultiAgentOrchestrator(collection)

# Arızalı bir kaydı seç
test_sample = df[df['label'] == 1].iloc[0].to_dict()

print("="*60)
print("  MULTI-AGENT PREDICTIVE MAINTENANCE PIPELINE")
print("="*60)
print(f"Test Verisi: Makine {test_sample['machine_id']} @ {test_sample['timestamp']}\n")

# Pipeline'ı çalıştır
result = orchestrator.run(test_sample)

  MULTI-AGENT PREDICTIVE MAINTENANCE PIPELINE
Test Verisi: Makine M01 @ 2024-07-01 08:01:00

🔵 Adım 1/3: SensorAnalyst çalışıyor...
   ✅ Tamamlandı (0.000s) → Risk: CRITICAL
🟡 Adım 2/3: KnowledgeRetriever çalışıyor...
   ✅ Tamamlandı (0.015s) → 3 doküman bulundu
🔴 Adım 3/3: MaintenancePlanner çalışıyor (LLM çağrısı)...
   ✅ Tamamlandı (4.082s)

🏁 Pipeline tamamlandı! Toplam süre: 4.10s


In [33]:
# Final raporu görüntüle
print("="*60)
print("  📋 FİNAL BAKIM RAPORU (MaintenancePlanner Çıktısı)")
print("="*60)
print(result['final_report'])

  📋 FİNAL BAKIM RAPORU (MaintenancePlanner Çıktısı)
**DURUM ÖZETİ**
Makine M01'in genel durumu kritik olarak değerlendirilmektedir. Sensör analiz sonuçları, makinenin vibration, acoustic, temperature ve current parametrelerinde anormal değerler gösterdiğini ortaya koymaktadır.

**TESPİT EDİLEN SORUNLAR**
1. **Vibration**: Makinenin vibration değeri 1.398 olarak ölçülmüş olup, bu değer kritik eşik olan 1.3'un üzerinde bulunmaktadır. Bu durum, makinede mekanik dengesizlik veya diğer yapısal sorunların varlığını işaret etmektedir.
2. **Acoustic**: Acoustic değer 0.834 olarak ölçülmüş olup, bu değer elevated olarak değerlendirilmektedir. Bu durum, rulman yataklarındaki yüksek frekanslı sürtünme seslerinin arttığını göstermektedir.
3. **Temperature**: Makinenin sıcaklık değeri 76.2°C olarak ölçülmüş olup, bu değer warning olarak değerlendirilmektedir. Bu durum, motor sargı ve yüzey sıcaklığının normal aralığın üzerinde olduğunu göstermektedir.
4. **Current**: Makinenin elektriksel yükü 15.0

In [35]:
# Execution Log (Performans Tablosu)
print("\n📊 Pipeline Performans Günlüğü:")
print("-"*50)
print(f"{'Ajan':<25} {'Süre (s)':<12} {'Durum':<8}")
print("-"*50)
for log in result['execution_log']:
    print(f"{log['agent']:<25} {log['duration_sec']:<12} {log['status']:<8}")
print("-"*50)
print(f"{'TOPLAM':<25} {result['total_duration_sec']:<12}")


📊 Pipeline Performans Günlüğü:
--------------------------------------------------
Ajan                      Süre (s)     Durum   
--------------------------------------------------
SensorAnalyst             0.0          OK      
KnowledgeRetriever        0.015        OK      
MaintenancePlanner        4.082        OK      
--------------------------------------------------
TOPLAM                    4.1         


## 3.8. Toplu Analiz: Tüm Arızalı Kayıtlar

Sistemi birden fazla arızalı kayıt üzerinde çalıştırarak:
1. Farklı arıza tiplerini doğru tespit edip etmediğini doğrularız
2. Ortalama pipeline süresini ölçeriz
3. Sonuçları karşılaştırmalı bir tablo halinde sunarız

In [38]:
# Arızalı kayıtlardan 5 farklı örnek seç (API kotasını korumak için sınırlı tutuyoruz)
failure_samples = df[df['label'] == 1].sample(n=5, random_state=42)

batch_results = []

print("="*60)
print("  TOPLU ANALİZ BAŞLIYOR (5 Arızalı Kayıt)")
print("="*60)

for idx, (_, row) in enumerate(failure_samples.iterrows()):
    print(f"\n{'─'*60}")
    print(f"📌 Kayıt {idx+1}/5: Makine {row['machine_id']} @ {row['timestamp']}")
    print(f"{'─'*60}")
    
    sample = row.to_dict()
    result = orchestrator.run(sample)
    batch_results.append(result)
    
    # Kısa özet göster
    risk = result['anomaly_report']['overall_risk']
    modes = [f['mode'] for f in result['anomaly_report']['detected_failure_modes']]
    print(f"   → Risk: {risk} | Arıza Modları: {modes if modes else 'Yok'}")

print(f"\n{'='*60}")
print(f"✅ Toplu analiz tamamlandı! {len(batch_results)} kayıt işlendi.")

  TOPLU ANALİZ BAŞLIYOR (5 Arızalı Kayıt)

────────────────────────────────────────────────────────────
📌 Kayıt 1/5: Makine M02 @ 2024-07-01 12:18:00
────────────────────────────────────────────────────────────
🔵 Adım 1/3: SensorAnalyst çalışıyor...
   ✅ Tamamlandı (0.000s) → Risk: CRITICAL
🟡 Adım 2/3: KnowledgeRetriever çalışıyor...
   ✅ Tamamlandı (0.033s) → 3 doküman bulundu
🔴 Adım 3/3: MaintenancePlanner çalışıyor (LLM çağrısı)...
   ✅ Tamamlandı (4.151s)

🏁 Pipeline tamamlandı! Toplam süre: 4.18s
   → Risk: CRITICAL | Arıza Modları: Yok

────────────────────────────────────────────────────────────
📌 Kayıt 2/5: Makine M01 @ 2024-07-01 10:17:00
────────────────────────────────────────────────────────────
🔵 Adım 1/3: SensorAnalyst çalışıyor...
   ✅ Tamamlandı (0.000s) → Risk: WARNING
🟡 Adım 2/3: KnowledgeRetriever çalışıyor...
   ✅ Tamamlandı (0.030s) → 3 doküman bulundu
🔴 Adım 3/3: MaintenancePlanner çalışıyor (LLM çağrısı)...
   ✅ Tamamlandı (4.360s)

🏁 Pipeline tamamlandı! Toplam 

In [39]:
# Karşılaştırmalı Özet Tablosu
print("\n📊 Karşılaştırmalı Sonuç Tablosu:")
print("="*90)
print(f"{'Makine':<8} {'Zaman':<22} {'Risk':<10} {'Arıza Modu':<35} {'Süre (s)':<10}")
print("-"*90)

for r in batch_results:
    machine = r['anomaly_report']['machine_id']
    ts = r['anomaly_report']['timestamp']
    risk = r['anomaly_report']['overall_risk']
    modes = ", ".join([f['mode'] for f in r['anomaly_report']['detected_failure_modes']]) or "Belirsiz"
    duration = r['total_duration_sec']
    print(f"{machine:<8} {ts:<22} {risk:<10} {modes:<35} {duration:<10}")

print("="*90)
avg_time = np.mean([r['total_duration_sec'] for r in batch_results])
print(f"Ortalama Pipeline Süresi: {avg_time:.2f}s")


📊 Karşılaştırmalı Sonuç Tablosu:
Makine   Zaman                  Risk       Arıza Modu                          Süre (s)  
------------------------------------------------------------------------------------------
M02      2024-07-01 12:18:00    CRITICAL   Belirsiz                            4.18      
M01      2024-07-01 10:17:00    WARNING    Belirsiz                            4.39      
M01      2024-07-01 12:27:00    CRITICAL   Belirsiz                            3.72      
M03      2024-07-01 12:54:00    WARNING    Belirsiz                            3.87      
M03      2024-07-01 16:10:00    WARNING    Belirsiz                            4.0       
Ortalama Pipeline Süresi: 4.03s


## 3.9. Sonuç ve Sonraki Adımlar

### Bu Notebook'ta Ne Yaptık?
- **3 uzman ajan** tanımladık, her biri farklı bir sorumluluk alanına sahip
- **Gateway Pattern** ile merkezi LLM erişimi kurduk (provider-agnostic)
- **Orchestrator** ile ajanları koordine ettik ve performans ölçümü yaptık
- Toplu analiz ile sistemin tutarlılığını doğruladık

### Mimari Kararlarımızın Gerekçeleri (AI Engineering Notları)
| Karar | Gerekçe |
| :--- | :--- |
| SensorAnalyst'te LLM kullanmamak | Eşik değerleri deterministiktir, LLM hallucination riski gereksizdir |
| KnowledgeRetriever'da semantic search | Tüm kılavuzu göndermek yerine sadece ilgili bölümleri getirmek maliyet ve doğruluk açısından üstündür |
| Sadece MaintenancePlanner'da LLM | LLM'in gücü yorum ve sentezde — onu en çok değer kattığı yerde kullanıyoruz |
| Gateway Pattern | Provider değişikliği (Groq → OpenAI → Gemini) tek satır değişiklik gerektirir |

### Sonraki Notebook (04)
- **Streamlit UI**: Kullanıcının makine seçip anlık analiz başlatabildiği interaktif dashboard
- **Rapor İndirme**: Üretilen raporların PDF olarak indirilmesi
- **Canlı Demo**: Hugging Face Spaces üzerinde deploy